# ⚡ Simple NVIDIA Nemotron-3-Nano-30B Inference via vLLM

This notebook provides a clean, single-prompt inference script for the **NVIDIA-Nemotron-3-Nano-30B** base model. It is directly inspired by the environment setup, package extraction, and vLLM configuration found in the competition baseline submissions.

## 🛠️ Step 1: Environment Setup & Package Unpacking

We uninstall the default PyTorch stack and unpack the custom Blackwell-optimized environment packages provided via Kaggle's metric utility script.

In [ ]:
import subprocess
import sys

commands = [
    "uv pip uninstall torch torchvision torchaudio",
    "tar -cf - -C /kaggle/usr/lib/notebooks/metric/nvidia_metric_utility_script . | tar -xf - -C /tmp",
    "chmod +x /tmp/triton/backends/nvidia/bin/ptxas",
    "chmod +x /tmp/triton/backends/nvidia/bin/ptxas-blackwell",
]

for cmd in commands:
    print(f"Running: {cmd}")
    subprocess.run(cmd, shell=True, check=True)

# Prepend the unpacked environment directory to python search path
sys.path.insert(0, "/tmp")

## 📥 Step 2: Configure Environment Variables & Load Model

In [ ]:
import os

# Set Triton and Transformers flags for offline execution
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TRITON_PTXAS_PATH"] = "/tmp/triton/backends/nvidia/bin/ptxas"

from vllm import LLM, SamplingParams
import kagglehub

# Download or locate the base model path
MODEL_PATH = kagglehub.model_download(
    "metric/nemotron-3-nano-30b-a3b-bf16/transformers/default"
)
print(f"Using model path: {MODEL_PATH}")

# Initialize vLLM Engine
llm = LLM(
    model=str(MODEL_PATH),
    tensor_parallel_size=1,
    max_num_seqs=64,
    gpu_memory_utilization=0.85,
    dtype="auto",
    max_model_len=8192,
    trust_remote_code=True,
    enable_lora=False,  # base model only
)

## 💬 Step 3: Run Single Prompt Inference

In [ ]:
# Set decoding parameters
sampling_params = SamplingParams(
    temperature=0.0,  # Greedier decoding
    top_p=1.0,
    max_tokens=2048,
)

# Define your prompt
prompt_text = "Convert the number 42 into Roman numerals."

# Format prompt using chat template
tokenizer = llm.get_tokenizer()
prompt = tokenizer.apply_chat_template(
    [{"role": "user", "content": prompt_text}],
    tokenize=False,
    add_generation_prompt=True,
)

print("Generating response...")
outputs = llm.generate([prompt], sampling_params=sampling_params)

# Print output
print("\n" + "="*40 + " RESPONSE " + "="*40)
print(outputs[0].outputs[0].text)
print("="*90)